In [5]:
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup, Comment
import re
import time

## Todo
* minimize requests (combine injuries with https://www.basketball-reference.com/teams/%s/2024.html call)
* add features (historical_performances, fatigue, see more below...)
* optimize weights using some algorithm (gradient descent)
* automize picking (using voice assistant or making drag and drop)
* more advanced shit like context of player's role,  
* EDA of all features

In [12]:
conversion_table = {
    "ATL": "Atlanta Hawks",
    "BOS": "Boston Celtics",
    "CHO": "Charlotte Hornets",
    "CHI": "Chicago Bulls",
    "CLE": "Cleveland Cavaliers",
    "DAL": "Dallas Mavericks",
    "DEN": "Denver Nuggets",
    "DET": "Detroit Pistons",
    "GSW": "Golden State Warriors",
    "HOU": "Houston Rockets",
    "IND": "Indiana Pacers",
    "LAC": "Los Angeles Clippers",
    "LAL": "Los Angeles Lakers",
    "MEM": "Memphis Grizzlies",
    "MIA": "Miami Heat",
    "MIL": "Milwaukee Bucks",
    "MIN": "Minnesota Timberwolves",
    "NOP": "New Orleans Pelicans",
    "NYK": "New York Knicks",
    "BRK": "Brooklyn Nets",
    "OKC": "Oklahoma City Thunder",
    "ORL": "Orlando Magic",
    "PHI": "Philadelphia 76ers",
    "PHO": "Phoenix Suns",
    "POR": "Portland Trail Blazers",
    "SAC": "Sacramento Kings",
    "TOR": "Toronto Raptors",
    "UTA": "Utah Jazz",
    "WAS": "Washington Wizards",
    "SAS": "San Antonio Spurs",
}

In [13]:
def normalize(data):
    total = sum(data)
    normalized_data = [x / total for x in data]
    return normalized_data

### Version 1.2: using pure linear weights to calculate odds

Offensive/defensive ratings //
Injuries //
Historical performances //
Home court advantage //
Fatigue (recency of games) //
Possibly momentum? //
Sentiment analysis? //
Advanced stats?

With each of these features, assign a weight of importance to each feature, then pull the real time data and calculate the total score



In [47]:
weights = [
    1.5,
    1.8,
    1.5,
    None,
    None,
    None,
    None,
]  # offdef_ratings, injuries, momentum                       historical_performances, fatigue, momentum, sentiment, advanced

In [40]:
class Team:
    def __init__(self, team_abbr):
        self.offdef_ratings = self.find_offdef_ratings(team_abbr)
        self.team_abbr = team_abbr
        self.df = pd.read_html(
            "https://www.basketball-reference.com/teams/%s/2024.html" % team_abbr
        )
        self.sleep = time.sleep(4) # sleep between two calls 
        self.game_df = pd.read_html(
            "https://www.basketball-reference.com/teams/%s/2024_games.html" % team_abbr
        )
        self.injuries = self.find_injuries()
        self.player_scores = self.calculate_player_scores()
        self.team_score_with, self.team_score_without = self.calculate_team_score()
        self.momentum = self.calculate_momentum()

    def calculate_player_scores(self):
        """
        return (dict): player --> sum of weights
        """
        player_weights = [
            0.2,
            0.2,
            0.2,
            0.2,
            0.2,
        ]  # points, assists, rebounds, win shares, value over replacement player

        player_scores = {}

        for index, player in self.df[1].iterrows():  # pts, ast, rebs
            name = player["Player"]
            player_scores[name] = []
            player_scores[name].append(player["PTS"])
            player_scores[name].append(player["AST"])
            player_scores[name].append(player["TRB"])

        for index, player in self.df[3].iterrows():
            name = player["Player"]
            player_scores[name].append(player["BPM"])
            player_scores[name].append(player["VORP"])

        scores = []
        for player in player_scores:
            for i in range(len(player_scores[player])):
                player_scores[player][i] = player_scores[player][i] * player_weights[i]
            player_scores[player] = sum(player_scores[player])
            scores.append(player_scores[player])
        scores = normalize(scores)
        for i, player in enumerate(player_scores):
            player_scores[player] = scores[i]

        return player_scores


    def calculate_importance(self, player):
        """
        player (pd.DataFrame) : dataframe of player on/off stats
        return ()
        """
        pass
        
    def calculate_team_score(self):
        """
        players_scores (dict): player: value
        injuries (list) : [[out players], [day to day players]]
        return (float): team score with, team score without
        """

        team_score_with = 1  # consider day to day players as out
        team_score_without = 1  # don't consider day to day players as out

        for lst in self.injuries:
            for player in lst:
                if (
                    player not in self.player_scores
                ):  # for whatever reason, player doesnt show up on roster/injury report
                    pass
                else:
                    score = self.player_scores[player]
                    team_score_with -= score

        for player in self.injuries[0]:
            if player not in self.player_scores:
                pass
            else:
                score = self.player_scores[player]
                team_score_without -= score
        if len(self.injuries[1]) == 0:
            return team_score_with, team_score_without
        else:
            print(
                "Day to Day Players (check for most recent update): ", self.injuries[1]
            )
            return team_score_with, team_score_without

    def find_injuries(self):
        """
        return (dict): team: [[out players], [day to day players]]
        """
        time.sleep(4)
        df = pd.read_html("https://www.basketball-reference.com/friv/injuries.fcgi")
        injuries = {}
        for index, player in df[0].iterrows():
            if player["Team"] not in injuries:
                if "Day To Day" in player["Description"].split("-")[0]:
                    injuries[player["Team"]] = [[], [player["Player"]]]
                else:
                    injuries[player["Team"]] = [[player["Player"]], []]
            else:
                if "Day To Day" in player["Description"].split("-")[0]:
                    injuries[player["Team"]][1].append(player["Player"])
                else:  # player is out
                    injuries[player["Team"]][0].append(player["Player"])
        if conversion_table[self.team_abbr] in injuries:
            return injuries[conversion_table[self.team_abbr]]
        else:
            return [[], []]

    def find_offdef_ratings(self, team_abbr):
        """
        team_abbr (str) : team abbreivation (CHI, BOS, LAL)
        return (list[float]) : [off_rtg, def_rtg]
        """
        constant = 200  # for inverse relationship for defensive ratings
        url = "https://www.basketball-reference.com/teams/%s/2024.html" % team_abbr
        r = requests.get(url)
        soup = BeautifulSoup(r.text, "html.parser")
        tag = soup.find("div", {"id": "all_team_misc"})

        for element in tag(text=lambda text: isinstance(text, Comment)):
            soup = BeautifulSoup(element, "html.parser")

        rtgs = soup.find_all("td", {"data-stat": ["off_rtg", "def_rtg"]})
        rtgs = np.log2(
            [float(rtgs[0].get_text()), constant - float(rtgs[1].get_text())]
        )
        time.sleep(4)
        print(rtgs)
        return rtgs

    def calculate_momentum(self):
        """
        return (float) : win percentage of last x number of games 
        """
        df = self.game_df[0]
        df = df[df["Unnamed: 7"].notna()]
        games_prior = 7 # number of games prior 
        last_games = df["Unnamed: 7"].tail(games_prior).tolist() 
        games_won = 0
        for game_result in last_games:
            if game_result == "W":
                games_won += 1
        win_percentage = games_won / games_prior
        
        return win_percentage

    def pipeline(self):
        """
        return (list): [offdef_ratings, injuries, historical_performances, fatigue, homecourt_advantage]
        """
        data = [
            sum(self.offdef_ratings) * weights[0],
            self.team_abbr,
            self.team_score_with * weights[1],
            self.team_score_without * weights[1],
            self.momentum * weights[2]
        ]

        return data

In [46]:
class Game:
    def __init__(self, home_team, away_team):
        self.home_team = home_team
        self.away_team = away_team

    def predict(self):
        """
        include_day_to_day (boolean) : decide whether to consider day to day players as out
        """
        home_team_sum_with = (
            sum(self.home_team.offdef_ratings) * weights[0]
            + self.home_team.team_score_with * weights[1]
            + 0.3
            + self.home_team.momentum * weights[2]
        )
        away_team_sum_with = (
            sum(self.away_team.offdef_ratings) * weights[0]
            + self.away_team.team_score_with * weights[1]
            + self.away_team.momentum * weights[2]
        )
        home_team_sum_without = (
            sum(self.home_team.offdef_ratings) * weights[0]
            + self.home_team.team_score_without * weights[1]
            + 0.3
            + self.home_team.momentum * weights[2]
        )
        away_team_sum_without = (
            sum(self.away_team.offdef_ratings) * weights[0]
            + self.away_team.team_score_without * weights[1]
            + self.away_team.momentum * weights[2]
        )

        if home_team_sum_with > away_team_sum_with:
            print("Winner with Day to Day Players out: " + self.home_team.team_abbr)
            print(
                self.home_team.team_abbr
                + ": "
                + str(home_team_sum_with)
                + " vs "
                + self.away_team.team_abbr
                + ": "
                + str(away_team_sum_with)
            )
        else:
            print("Winner with Day to Day Players out: " + self.away_team.team_abbr)
            print(
                self.home_team.team_abbr
                + ": "
                + str(home_team_sum_with)
                + " vs "
                + self.away_team.team_abbr
                + ": "
                + str(away_team_sum_with)
            )

        print("")

        if home_team_sum_without > away_team_sum_without:
            print("Winner with Day to Day Players out: " + self.home_team.team_abbr)
            print(
                self.home_team.team_abbr
                + ": "
                + str(home_team_sum_without)
                + " vs "
                + self.away_team.team_abbr
                + ": "
                + str(away_team_sum_without)
            )
        else:
            print("Winner with Day to Day Players out: " + self.away_team.team_abbr)
            print(
                self.home_team.team_abbr
                + ": "
                + str(home_team_sum_without)
                + " vs "
                + self.away_team.team_abbr
                + ": "
                + str(away_team_sum_without)
            )

### Outcome Testing

In [26]:
sixers = Team("PHI")

[6.93310047 6.43629512]


In [50]:
sixers.player_scores

{'Tyrese Maxey': 0.1920928233777396,
 'Tobias Harris': 0.10872367855608078,
 'Joel Embiid': 0.27847013321873654,
 "De'Anthony Melton": 0.09282337773957884,
 'Kelly Oubre Jr.': 0.09024495058014612,
 'Nicolas Batum': 0.06532015470562957,
 'P.J. Tucker': 0.012462397937258275,
 'Patrick Beverley': 0.0416845724108294,
 'Robert Covington': 0.0631714654061023,
 'Paul Reed': 0.04254404813064031,
 'Danuel House Jr.': 0.015900300816501935,
 'Marcus Morris': 0.03223033949290933,
 'Jaden Springer': 0.0012892135797163793,
 'Danny Green': -0.02105715513536743,
 'Furkan Korkmaz': 0.0017189514396218322,
 'Mo Bamba': 0.02406532015470563,
 'KJ Martin': -0.024495058014611087,
 'Filip Petrušev': -0.017189514396218308}

In [45]:
sixers.injuries

[[], []]

In [48]:
sixers = Team("PHI")
pistons = Team("DET")
Game(pistons, sixers).predict()

[6.93310047 6.43629512]
[6.76022095 6.34872815]
Day to Day Players (check for most recent update):  ['Marvin Bagley III', 'Jalen Duren']
Winner with Day to Day Players out: PHI
DET: 21.390366656227727 vs PHI: 22.925521963584668

Winner with Day to Day Players out: PHI
DET: 21.76342365104638 vs PHI: 22.925521963584668


In [24]:
sixers.pipeline()

[13.36939559477073, 'PHI', 1, 1, 0.7142857142857143]

In [49]:
hawks = Team("ATL")
raptors = Team("TOR")
Game(raptors, hawks).predict()

[6.90086681 6.32192809]
Day to Day Players (check for most recent update):  ["De'Andre Hunter"]
[6.81378119 6.41953889]
Day to Day Players (check for most recent update):  ['Chris Boucher', 'Otto Porter Jr.']
Winner with Day to Day Players out: TOR
TOR: 22.090932505048617 vs ATL: 21.78596920962261

Winner with Day to Day Players out: TOR
TOR: 22.16426583838195 vs ATL: 22.01176651649717


In [17]:
hornets = Team("CHO")
heat = Team("MIA")
Game(heat, hornets).predict()

Day to Day Players (check for most recent update):  ['Mark Williams']
Winner with Day to Day Players out: MIA
MIA: 5.168097497360562 vs CHO: 4.893118591057709

Winner with Day to Day Players out: MIA
MIA: 5.168097497360562 vs CHO: 4.927700264364482


In [18]:
lakers = Team("LAL")
spurs = Team("SAS")
Game(spurs, lakers).predict()

Winner with Day to Day Players out: SAS
SAS: 5.108990846827633 vs LAL: 5.0835346673352255

Winner with Day to Day Players out: SAS
SAS: 5.108990846827633 vs LAL: 5.0835346673352255


In [19]:
pacers = Team("IND")
bucks = Team("MIL")
Game(bucks, pacers).predict()

Winner with Day to Day Players out: MIL
MIL: 5.225303943482405 vs IND: 5.054189173658766

Winner with Day to Day Players out: MIL
MIL: 5.225303943482405 vs IND: 5.054189173658766


In [20]:
grizzlies = Team("MEM")
rockets = Team("HOU")
Game(rockets, grizzlies).predict()

Day to Day Players (check for most recent update):  ['Reggie Bullock', 'Tari Eason', 'Amen Thompson']
Winner with Day to Day Players out: HOU
HOU: 5.264232626923988 vs MEM: 4.969154616212659

Winner with Day to Day Players out: HOU
HOU: 5.300886936469864 vs MEM: 4.969154616212659


In [21]:
knicks = Team("NYK")
jazz = Team("UTA")
Game(jazz, knicks).predict()

Day to Day Players (check for most recent update):  ['Immanuel Quickley']
Day to Day Players (check for most recent update):  ['John Collins', 'Walker Kessler', 'Lauri Markkanen']
Winner with Day to Day Players out: UTA
UTA: 5.018825496936471 vs NYK: 4.985152860732536

Winner with Day to Day Players out: UTA
UTA: 5.184556896684264 vs NYK: 5.0448387246068815


In [22]:
nets = Team("BRK")
suns = Team("PHO")
Game(suns, nets).predict()

Winner with Day to Day Players out: PHO
PHO: 5.153225694535496 vs BRK: 5.0188977202917044

Winner with Day to Day Players out: PHO
PHO: 5.153225694535496 vs BRK: 5.0188977202917044


### Code Testing

In [4]:
df = pd.read_html("https://www.basketball-reference.com/teams/GSW/2024/on-off/")

In [15]:
df[0]

Unnamed: 0_level_0 Unnamed: 1_level_0 Unnamed: 2_level_0   Team         \
               Player              Split                 MP   eFG%   ORB%   
0       Stephen Curry           On Court                677   .550   28.4   
1                 NaN          Off Court                390   .507   28.7   
2                 NaN           On − Off                63%  +.043   -0.3   
3                 NaN                NaN                NaN    NaN    NaN   
4       Klay Thompson           On Court                634   .547   27.7   
..                ...                ...                ...    ...    ...   
58                NaN           On − Off                 1%  -.114  +21.7   
59             Player              Split                 MP   eFG%   ORB%   
60    Jerome Robinson           On Court                  5   .450   66.7   
61                NaN          Off Court               1062   .534   28.3   
62                NaN           On − Off                 0%  -.084  +38.4   

                                      ... Difference                       \
     DRB%   TRB%   AST%  STL%   BLK%  ...       eFG%   ORB%   DRB%   TRB%   
0    73.9   51.2   69.3   5.8    5.3  ...      +.008   +2.3   +2.3   +2.3   
1    78.5   52.8   66.6   8.7    8.1  ...      +.002   +7.2   +7.2   +5.6   
2    -4.6   -1.6   +2.7  -2.9   -2.8  ...      +.006   -4.9   -4.9   -3.3   
3     NaN    NaN    NaN   NaN    NaN  ...        NaN    NaN    NaN    NaN   
4    73.8   50.6   68.9   5.6    5.7  ...      +.004   +1.6   +1.6   +1.2   
..    ...    ...    ...   ...    ...  ...        ...    ...    ...    ...   
58  +12.0  +13.3  -25.6  -6.9  +27.3  ...      +.010  +33.7  +33.7  +26.7   
59   DRB%   TRB%   AST%  STL%   BLK%  ...       eFG%   ORB%   DRB%   TRB%   
60   83.3   75.0   25.0   0.0   42.9  ...      +.061  +50.0  +50.0  +50.0   
61   75.6   51.7   68.5   6.9    6.1  ...      +.005   +3.9   +3.9   +3.3   
62   +7.7  +23.3  -43.5  -6.9  +36.8  ...      +.056  +46.1  +46.1  +46.7   

                                              
     AST%   STL%   BLK%   TOV%   Pace   ORtg  
0   +10.4   -3.1   -2.2   +5.1   +0.9   -2.0  
1    +5.9   +1.7   -1.7   -1.1   -2.0   +4.8  
2    +4.5   -4.8   -0.5   +6.2   +2.9   -6.8  
3     NaN    NaN    NaN    NaN    NaN    NaN  
4   +10.0   -1.7   -2.2   +3.6   -0.2   -1.2  
..    ...    ...    ...    ...    ...    ...  
58  -16.1  -10.6  +35.8   +3.1   +0.1   +5.4  
59   AST%   STL%   BLK%   TOV%   Pace   ORtg  
60   -8.3  -22.2  +42.9  +22.2  -10.3  +11.1  
61   +8.9   -1.2   -2.4   +2.8   -0.1   +0.5  
62  -17.2  -21.0  +45.3  +19.4  -10.2  +10.6  

[63 rows x 33 columns]

In [96]:
df

,G,Date,Start (ET),Unnamed: 3,Unnamed: 4,Unnamed: 5,Opponent,Unnamed: 7,Unnamed: 8,Tm,Opp,W,L,Streak,Notes
0,1,"Wed, Oct 25, 2023",9:00p,NaN,Box Score,@,Utah Jazz,W,NaN,130,114,1,0,W 1,NaN
1,2,"Fri, Oct 27, 2023",10:00p,NaN,Box Score,NaN,Golden State Warriors,L,NaN,114,122,1,1,L 1,NaN
2,3,"Sun, Oct 29, 2023",9:00p,NaN,Box Score,NaN,Los Angeles Lakers,W,OT,132,127,2,1,W 1,NaN
3,4,"Wed, Nov 1, 2023",10:00p,NaN,Box Score,@,Golden State Warriors,L,NaN,101,102,2,2,L 1,NaN
4,5,"Sat, Nov 4, 2023",8:00p,NaN,Box Score,@,Houston Rockets,L,NaN,89,107,2,3,L 2,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81,79,"Tue, Apr 9, 2024",8:00p,NaN,NaN,@,Oklahoma City Thunder,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
82,80,"Thu, Apr 11, 2024",10:00p,NaN,NaN,NaN,New Orleans Pelicans,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
83,G,Date,Start (ET),NaN,NaN,NaN,Opponent,NaN,NaN,Tm,Opp,W,L,Streak,Notes
84,81,"Fri, Apr 12, 2024",10:30p,NaN,NaN,NaN,Phoenix Suns,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [97]:
df = df[df["Unnamed: 7"].notna()]

In [102]:
df["Unnamed: 7"].tail(7).tolist()

['W', 'W', 'L', 'W', 'L', 'W', 'W']

In [ ]:
df = pd.read_html("https://www.basketball-reference.com/friv/injuries.fcgi")

## Find players that are out (cbssports method)

In [2]:
def find_team_name(tag):
    pattern = r'>(.*?)<\/a>'
    team_name = str(tag.find_all(class_ = "TeamName")[0])
    team_name = re.search(pattern, team_name).group(1)
    team_name = team_name.split('>', 1)[-1]
    return team_name

In [ ]:
# def find_injury_time(tag):
#     element = str(tag).split("width: 40%;")[1]
#     start_index = element.find('">') + 2
#     end_index = element.find('</td>')
#     injury_time = element[start_index:end_index].strip()
#     if injury_time != "Game Time Decision":
#         return "Out"
#     else:
#         return "Questionable"

In [8]:
def find_injuries():
    injuries = {}
    r = requests.get('https://www.cbssports.com/nba/injuries/')
    soup = BeautifulSoup(r.text, 'html.parser')
    teams = soup.find_all(class_ = 'TableBaseWrapper')
    for team_element in teams:
        team_name = find_team_name(team_element)
        injuries[team_name] = []
        for report_element in team_element.find_all('tr', class_='TableBase-bodyTr'):
            player_name = report_element.find('span', class_='CellPlayerName--long').text
            # injury_time = find_injury_time(report_element)
            injuries[team_name].append(player_name)
    return injuries


In [9]:
injuries = find_injuries()

In [10]:
injuries.keys()

{'Atlanta': ['Trae Young',
  'Vit Krejci',
  "De'Andre Hunter",
  'Wesley Matthews',
  'Mouhamed Gueye'],
 'Boston': ['Al Horford', 'Jrue Holiday'],
 'Brooklyn': ['Dariq Whitehead', "Day'Ron Sharpe", 'Ben Simmons'],
 'Charlotte': ['Nick Richards',
  'LaMelo Ball',
  'Mark Williams',
  'Gordon Hayward'],
 'Chicago': ['Zach LaVine', 'Torrey Craig', 'Lonzo Ball'],
 'Cleveland': ['Ty Jerome', 'Caris LeVert', 'Darius Garland', 'Evan Mobley'],
 'Dallas': ['Josh Green', 'Dante Exum'],
 'Denver': ['Julian Strawther', 'Aaron Gordon', 'Vlatko Cancar'],
 'Detroit': ['Isaiah Stewart',
  'Malcolm Cazalon',
  'Cade Cunningham',
  'Monte Morris'],
 'Golden St.': ['Moses Moody', 'Chris Paul', 'Gary Payton II'],
 'Houston': ['Victor Oladipo', 'Reggie Bullock', 'Tari Eason'],
 'Indiana': ['Tyrese Haliburton', 'Andrew Nembhard', 'Isaiah Jackson'],
 'L.A. Clippers': ['Moussa Diabate', 'Xavier Moon', 'Ivica Zubac'],
 'L.A. Lakers': ['LeBron James',
  'Cam Reddish',
  'Taurean Prince',
  'Anthony Davis',
  